In [105]:
import pandas as pd

df = pd.read_csv("Maharashtra_AllCities_Hospital_Dataset.csv")

print("Shape:", df.shape)
print(df.head())

Shape: (851, 47)
  Hospital_ID     District    City        State City_Type  \
0     MH-1001  Mumbai City  Colaba  Maharashtra     Urban   
1     MH-1002  Mumbai City  Colaba  Maharashtra     Urban   
2     MH-1003  Mumbai City  Colaba  Maharashtra     Urban   
3     MH-1004  Mumbai City  Colaba  Maharashtra     Urban   
4     MH-1005  Mumbai City    Fort  Maharashtra     Urban   

                   Hospital_Name Reference_Period_AU595  Total_Beds  ICU_Beds  \
0        City Health Care Colaba             31.08.2020         119        33   
1  Central Hospital & ICU Colaba             31.08.2020         133        32   
2         Sparsh Hospital Colaba             31.08.2020         137        52   
3    Fortis Trauma Centre Colaba             31.08.2020          94        41   
4   Multispecialty Hospital Fort             31.08.2020         119        31   

   Non_ICU_Beds  ...  Doctors_Orthopedic  Doctors_Pediatrician  Total_Doctors  \
0            86  ...                   2        

In [106]:
import numpy as np

np.random.seed(42)
df['beds_occupied'] = (
    df['Total_Beds'] * np.random.uniform(0.3, 0.95, size=len(df))
).astype(int)

df['utilization_rate'] = df['beds_occupied'] / df['Total_Beds'].replace(0, 1)

df['surge_status'] = (df['utilization_rate'] > 0.75).astype(int)

print(df['surge_status'].value_counts())


surge_status
0    652
1    199
Name: count, dtype: int64


In [107]:
df['icu_ratio'] = df['ICU_Beds'] / df['Total_Beds'].replace(0, 1)

df['nonicu_ratio'] = df['Non_ICU_Beds'] / df['Total_Beds'].replace(0, 1)

df['doctor_bed_ratio'] = df['Total_Doctors'] / df['Total_Beds'].replace(0, 1)

df['ventilator_ratio'] = df['Ventilators'] / df['Total_Beds'].replace(0, 1)

df['room_density'] = df['Total_Rooms'] / df['Total_Beds'].replace(0, 1)

df['emergency_ratio'] = df['Emergency_Rooms'] / df['Total_Rooms'].replace(0, 1)

df = df.fillna(0)

In [108]:
from sklearn.preprocessing import LabelEncoder

df = df.drop(['Hospital_ID', 'Hospital_Name', 'Reference_Period_AU595'], axis=1)

le_city = LabelEncoder()
le_district = LabelEncoder()
le_state = LabelEncoder()
le_city_type = LabelEncoder()

df['City_enc'] = le_city.fit_transform(df['City'])
df['District_enc'] = le_district.fit_transform(df['District'])
df['State_enc'] = le_state.fit_transform(df['State'])
df['City_Type_enc'] = le_city_type.fit_transform(df['City_Type'])

print(df[['City', 'City_enc']].head())

     City  City_enc
0  Colaba        63
1  Colaba        63
2  Colaba        63
3  Colaba        63
4    Fort        86


In [109]:
df = df.drop(['City', 'District', 'State', 'City_Type'], axis=1)

In [110]:
FEATURES = [
    'Total_Beds', 'Available_Beds', 'beds_occupied',
    'ICU_Beds', 'Non_ICU_Beds',
    'Total_Doctors', 'Ventilators',

    'utilization_rate',
    'icu_ratio',
    'nonicu_ratio',
    'doctor_bed_ratio',
    'ventilator_ratio',
    'room_density',
    'emergency_ratio',

    'City_enc',
    'District_enc',
    'State_enc',
    'City_Type_enc'
]

X = df[FEATURES]
y = df['surge_status']

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (851, 18)
y shape: (851,)


In [111]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain distribution:")
print(y_train.value_counts())

print("\nTest distribution:")
print(y_test.value_counts())

Train shape: (680, 18)
Test shape: (171, 18)

Train distribution:
surge_status
0    521
1    159
Name: count, dtype: int64

Test distribution:
surge_status
0    131
1     40
Name: count, dtype: int64


In [112]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight='balanced',
    random_state=42
)

clf.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [113]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

ROC-AUC: 1.0

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       131
           1       1.00      1.00      1.00        40

    accuracy                           1.00       171
   macro avg       1.00      1.00      1.00       171
weighted avg       1.00      1.00      1.00       171



Bed Occupancy Model

In [114]:
y_reg = df['beds_occupied']

In [115]:
FEATURES_REG = [
    'Total_Beds', 'Available_Beds',
    'ICU_Beds', 'Non_ICU_Beds',
    'Total_Doctors', 'Ventilators',

    'icu_ratio',
    'nonicu_ratio',
    'doctor_bed_ratio',
    'ventilator_ratio',
    'room_density',
    'emergency_ratio',

    'City_enc',
    'District_enc',
    'State_enc',
    'City_Type_enc'
]

X_reg = df[FEATURES_REG]

In [116]:
from sklearn.model_selection import train_test_split

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg,
    test_size=0.2,
    random_state=42
)

In [117]:
from sklearn.ensemble import GradientBoostingRegressor

reg = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

reg.fit(Xr_train, yr_train)

print("Regression model trained!")

Regression model trained!


In [118]:
from sklearn.metrics import mean_absolute_error, r2_score

yr_pred = reg.predict(Xr_test)

print("MAE:", mean_absolute_error(yr_test, yr_pred))
print("R2 Score:", r2_score(yr_test, yr_pred))

MAE: 5.622954313557005
R2 Score: 0.786592948614513


In [119]:
predicted_beds = yr_pred
total_beds = Xr_test['Total_Beds']

utilization = predicted_beds / total_beds
print(utilization[:5])

66     0.482702
316    0.556121
658    0.513323
797    0.575053
398    0.619031
Name: Total_Beds, dtype: float64


staff


In [120]:
import pandas as pd

df = pd.read_csv("Maharashtra_AllCities_Hospital_Dataset.csv")

print(df.shape)
print(df.head())

(851, 47)
  Hospital_ID     District    City        State City_Type  \
0     MH-1001  Mumbai City  Colaba  Maharashtra     Urban   
1     MH-1002  Mumbai City  Colaba  Maharashtra     Urban   
2     MH-1003  Mumbai City  Colaba  Maharashtra     Urban   
3     MH-1004  Mumbai City  Colaba  Maharashtra     Urban   
4     MH-1005  Mumbai City    Fort  Maharashtra     Urban   

                   Hospital_Name Reference_Period_AU595  Total_Beds  ICU_Beds  \
0        City Health Care Colaba             31.08.2020         119        33   
1  Central Hospital & ICU Colaba             31.08.2020         133        32   
2         Sparsh Hospital Colaba             31.08.2020         137        52   
3    Fortis Trauma Centre Colaba             31.08.2020          94        41   
4   Multispecialty Hospital Fort             31.08.2020         119        31   

   Non_ICU_Beds  ...  Doctors_Orthopedic  Doctors_Pediatrician  Total_Doctors  \
0            86  ...                   2               

In [121]:
import numpy as np

np.random.seed(42)

# Simulate patient load (30% to 95% of beds)
df['patients'] = (
    df['Total_Beds'] * np.random.uniform(0.3, 0.95, size=len(df))
).astype(int)

print(df[['Total_Beds', 'patients']].head())

   Total_Beds  patients
0         119        64
1         133       122
2         137       106
3          94        64
4         119        47


In [122]:
import numpy as np

np.random.seed(42)

df['doctors_needed'] = (
    (df['patients'] / 10) +
    np.random.normal(0, 1, size=len(df))   # noise
).round().clip(lower=1)

df['nurses_needed'] = (
    (df['patients'] / 5) +
    np.random.normal(0, 2, size=len(df))
).round().clip(lower=1)

df['surgeons_needed'] = (
    (df['patients'] / 25) +
    np.random.normal(0, 1, size=len(df))
).round().clip(lower=1)

In [123]:
df['urban_pressure'] = (df['City_Type'] == 'Urban').astype(int)

df['equipment_strength'] = (
    df['Ventilators'] +
    df['Oxygen_Concentrators']
) / df['Total_Beds'].replace(0, 1)

In [124]:
df['patient_doctor_pressure'] = df['patients'] / df['Total_Doctors'].replace(0, 1)

In [125]:
df['utilization'] = df['patients'] / df['Total_Beds'].replace(0, 1)

df['icu_ratio'] = df['ICU_Beds'] / df['Total_Beds'].replace(0, 1)

df['doctor_bed_ratio'] = df['Total_Doctors'] / df['Total_Beds'].replace(0, 1)

df = df.fillna(0)

In [126]:
df['patients'] = (
    df['Total_Beds'] *
    np.random.uniform(0.3, 0.95, size=len(df)) *
    np.random.uniform(0.9, 1.1, size=len(df))  # extra variation
).astype(int)

In [127]:
from sklearn.preprocessing import LabelEncoder

le_city = LabelEncoder()
le_district = LabelEncoder()

df['City_enc'] = le_city.fit_transform(df['City'])
df['District_enc'] = le_district.fit_transform(df['District'])

In [128]:
FEATURES = [
    'Total_Beds',
    'utilization',
    'icu_ratio',
    'doctor_bed_ratio',
    'City_enc',
    'District_enc'
]

X = df[FEATURES]

y_doctors = df['doctors_needed']
y_nurses = df['nurses_needed']
y_surgeons = df['surgeons_needed']

In [129]:
from sklearn.model_selection import train_test_split

X_train, X_test, yd_train, yd_test = train_test_split(X, y_doctors, test_size=0.2, random_state=42)
_, _, yn_train, yn_test = train_test_split(X, y_nurses, test_size=0.2, random_state=42)
_, _, ys_train, ys_test = train_test_split(X, y_surgeons, test_size=0.2, random_state=42)

In [130]:
from sklearn.ensemble import GradientBoostingRegressor

model_doc = GradientBoostingRegressor().fit(X_train, yd_train)
model_nurse = GradientBoostingRegressor().fit(X_train, yn_train)
model_surg = GradientBoostingRegressor().fit(X_train, ys_train)

print("All models trained!")

All models trained!


In [131]:
from sklearn.metrics import mean_absolute_error

print("Doctors MAE:", mean_absolute_error(yd_test, model_doc.predict(X_test)))
print("Nurses MAE:", mean_absolute_error(yn_test, model_nurse.predict(X_test)))
print("Surgeons MAE:", mean_absolute_error(ys_test, model_surg.predict(X_test)))

Doctors MAE: 0.5487624083962492
Nurses MAE: 1.4572337102735993
Surgeons MAE: 0.36256834341530125


In [132]:
sample = X_test.iloc[0:1]

print("Predicted Doctors:", model_doc.predict(sample))
print("Predicted Nurses:", model_nurse.predict(sample))
print("Predicted Surgeons:", model_surg.predict(sample))

Predicted Doctors: [2.85045241]
Predicted Nurses: [4.79110497]
Predicted Surgeons: [1.11882304]


In [133]:
def hospital_pipeline(input_data):
    import pandas as pd
    
    df = pd.DataFrame([input_data])
    
    df['icu_ratio'] = df['ICU_Beds'] / df['Total_Beds']
    df['nonicu_ratio'] = df['Non_ICU_Beds'] / df['Total_Beds']
    df['doctor_bed_ratio'] = df['Total_Doctors'] / df['Total_Beds']
    df['ventilator_ratio'] = df['Ventilators'] / df['Total_Beds']
    df['room_density'] = df['Total_Rooms'] / df['Total_Beds']
    df['emergency_ratio'] = df['Emergency_Rooms'] / df['Total_Rooms']
    
    df = df.fillna(0)
    
    df['City_enc'] = le_city.transform(df['City'])
    df['District_enc'] = le_district.transform(df['District'])
    df['State_enc'] = le_state.transform(df['State'])
    df['City_Type_enc'] = le_city_type.transform(df['City_Type'])
    
    X_reg_input = df[[
        'Total_Beds', 'Available_Beds',
        'ICU_Beds', 'Non_ICU_Beds',
        'Total_Doctors', 'Ventilators',
        'icu_ratio', 'nonicu_ratio',
        'doctor_bed_ratio', 'ventilator_ratio',
        'room_density', 'emergency_ratio',
        'City_enc', 'District_enc',
        'State_enc', 'City_Type_enc'
    ]]
    
    predicted_beds = reg.predict(X_reg_input)[0]

    utilization = predicted_beds / df['Total_Beds'].values[0]

    df['utilization'] = utilization
    
    X_staff_input = df[[
        'Total_Beds',
        'utilization',
        'icu_ratio',
        'doctor_bed_ratio',
        'City_enc',
        'District_enc'
    ]]
    
    doctors = model_doc.predict(X_staff_input)[0]
    nurses = model_nurse.predict(X_staff_input)[0]
    surgeons = model_surg.predict(X_staff_input)[0]

    df['beds_occupied'] = predicted_beds
    df['utilization_rate'] = utilization

    X_clf_input = df[[
    'Total_Beds', 'Available_Beds',
    'beds_occupied',
    'ICU_Beds', 'Non_ICU_Beds',
    'Total_Doctors', 'Ventilators',
    'utilization_rate',
    'icu_ratio', 'nonicu_ratio',
    'doctor_bed_ratio', 'ventilator_ratio',
    'room_density', 'emergency_ratio',
    'City_enc', 'District_enc',
    'State_enc', 'City_Type_enc'
]]
    
    surge = clf.predict(X_clf_input)[0]

    return {
        "Beds Occupied": round(predicted_beds),
        "Utilization (%)": round(utilization * 100, 2),
        "Doctors Needed": round(doctors),
        "Nurses Needed": round(nurses),
        "Surgeons Needed": round(surgeons),
        "Surge Status": "YES" if surge == 1 else "NO"
    }

In [134]:
sample = {
    "Total_Beds": 200,
    "Available_Beds": 60,
    "ICU_Beds": 40,
    "Non_ICU_Beds": 160,
    "Total_Doctors": 25,
    "Ventilators": 20,
    "Total_Rooms": 120,
    "Emergency_Rooms": 12,
    
    "City": "Colaba",
    "District": "Mumbai City",
    "State": "Maharashtra",
    "City_Type": "Urban"
}

result = hospital_pipeline(sample)

for k, v in result.items():
    print(k, ":", v)

Beds Occupied : 72
Utilization (%) : 35.94
Doctors Needed : 5
Nurses Needed : 8
Surgeons Needed : 3
Surge Status : NO


OPD

In [135]:
np.random.seed(42)

df['patients'] = (
    df['Total_Beds'] *
    np.random.uniform(0.3, 0.95, size=len(df)) *
    np.random.uniform(0.9, 1.1, size=len(df))  # extra realism
).astype(int)

print(df[['Total_Beds', 'patients']].head())

   Total_Beds  patients
0         119        67
1         133       115
2         137       114
3          94        60
4         119        44


In [136]:
safe_doctors = df['Total_Doctors'].replace(0, 1)

df['wait_time'] = (
    (df['patients'] / safe_doctors) * 2.5 +   # load effect
    (df['patients'] / df['Total_Beds']) * 30  # utilization effect
)

df['wait_time'] += np.random.normal(0, 5, size=len(df))

df['wait_time'] = df['wait_time'].clip(lower=5, upper=120)

print(df[['patients', 'Total_Doctors', 'wait_time']].head())

   patients  Total_Doctors  wait_time
0        67             31  22.616354
1       115             40  23.250017
2       114             48  26.204327
3        60             42  21.999927
4        44             40   7.793963


In [137]:
df['utilization'] = df['patients'] / df['Total_Beds'].replace(0, 1)

df['patient_doctor_ratio'] = df['patients'] / df['Total_Doctors'].replace(0, 1)

df['icu_ratio'] = df['ICU_Beds'] / df['Total_Beds'].replace(0, 1)

df['room_density'] = df['Total_Rooms'] / df['Total_Beds'].replace(0, 1)

df['emergency_ratio'] = df['Emergency_Rooms'] / df['Total_Rooms'].replace(0, 1)

df = df.fillna(0)

In [138]:
from sklearn.preprocessing import LabelEncoder

le_city = LabelEncoder()
le_district = LabelEncoder()
le_state = LabelEncoder()
le_city_type = LabelEncoder()

df['City_enc'] = le_city.fit_transform(df['City'])
df['District_enc'] = le_district.fit_transform(df['District'])
df['State_enc'] = le_state.fit_transform(df['State'])
df['City_Type_enc'] = le_city_type.fit_transform(df['City_Type'])

In [139]:
FEATURES_WAIT = [
    'Total_Beds',
    'Available_Beds',
    'Total_Doctors',
    'patients',
    'utilization',
    'patient_doctor_ratio',
    'icu_ratio',
    'room_density',
    'emergency_ratio',
    'City_enc',
    'District_enc',
    'State_enc',
    'City_Type_enc'
]

X = df[FEATURES_WAIT]
y = df['wait_time']

In [140]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [141]:
from sklearn.ensemble import GradientBoostingRegressor

wait_model = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

wait_model.fit(X_train, y_train)

print("Wait time model trained!")

Wait time model trained!


In [142]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = wait_model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 4.3407126997152155
RMSE: 5.442446950003648
R2 Score: 0.5249181812697404


In [145]:
def predict_wait_time(input_data):
    import pandas as pd
    
    df_input = pd.DataFrame([input_data])
    
    # STEP 1 — Create patients (THIS FIXES ERROR)
    df_input['patients'] = (
        df_input['Total_Beds'] - df_input['Available_Beds']
    ).clip(lower=1)
    
    # STEP 2 — Features
    df_input['utilization'] = df_input['patients'] / df_input['Total_Beds']
    
    df_input['patient_doctor_ratio'] = (
        df_input['patients'] / df_input['Total_Doctors'].replace(0, 1)
    )
    
    df_input['icu_ratio'] = df_input['ICU_Beds'] / df_input['Total_Beds']
    
    df_input['room_density'] = df_input['Total_Rooms'] / df_input['Total_Beds']
    
    df_input['emergency_ratio'] = df_input['Emergency_Rooms'] / df_input['Total_Rooms']
    
    # Encoding
    df_input['City_enc'] = le_city.transform(df_input['City'])
    df_input['District_enc'] = le_district.transform(df_input['District'])
    df_input['State_enc'] = le_state.transform(df_input['State'])
    df_input['City_Type_enc'] = le_city_type.transform(df_input['City_Type'])
    
    df_input = df_input.fillna(0)
    
    X_input = df_input[FEATURES_WAIT]
    
    pred = wait_model.predict(X_input)[0]
    
    return {
    "Predicted Wait Time (mins)": round(float(pred), 2)
}

In [146]:
sample = {
    "Total_Beds": 150,
    "Available_Beds": 40,
    "ICU_Beds": 30,
    "Non_ICU_Beds": 120,
    "Total_Doctors": 3,
    "Ventilators": 10,
    "Total_Rooms": 80,
    "Emergency_Rooms": 10,
    "City": "Colaba",
    "District": "Mumbai City",
    "State": "Maharashtra",
    "City_Type": "Urban"
}

print(predict_wait_time(sample))

{'Predicted Wait Time (mins)': 23.4}


In [147]:
def full_hospital_pipeline(input_data):
    import pandas as pd
    
    df = pd.DataFrame([input_data])
    
    # =========================
    # STEP 1 — Feature Engineering
    # =========================
    df['icu_ratio'] = df['ICU_Beds'] / df['Total_Beds']
    df['nonicu_ratio'] = df['Non_ICU_Beds'] / df['Total_Beds']
    df['doctor_bed_ratio'] = df['Total_Doctors'] / df['Total_Beds']
    df['ventilator_ratio'] = df['Ventilators'] / df['Total_Beds']
    df['room_density'] = df['Total_Rooms'] / df['Total_Beds']
    df['emergency_ratio'] = df['Emergency_Rooms'] / df['Total_Rooms']
    
    df = df.fillna(0)
    
    # =========================
    # STEP 2 — Encoding
    # =========================
    df['City_enc'] = le_city.transform(df['City'])
    df['District_enc'] = le_district.transform(df['District'])
    df['State_enc'] = le_state.transform(df['State'])
    df['City_Type_enc'] = le_city_type.transform(df['City_Type'])
    
    # =========================
    # STEP 3 — MODEL 3 (Beds)
    # =========================
    X_reg_input = df[[
        'Total_Beds', 'Available_Beds',
        'ICU_Beds', 'Non_ICU_Beds',
        'Total_Doctors', 'Ventilators',
        'icu_ratio', 'nonicu_ratio',
        'doctor_bed_ratio', 'ventilator_ratio',
        'room_density', 'emergency_ratio',
        'City_enc', 'District_enc',
        'State_enc', 'City_Type_enc'
    ]]
    
    predicted_beds = reg.predict(X_reg_input)[0]
    
    # =========================
    # STEP 4 — Utilization
    # =========================
    utilization = predicted_beds / df['Total_Beds'].values[0]
    
    # =========================
    # STEP 5 — MODEL 2 (Staff)
    # =========================
    df['utilization'] = utilization
    
    X_staff_input = df[[
        'Total_Beds',
        'utilization',
        'icu_ratio',
        'doctor_bed_ratio',
        'City_enc',
        'District_enc'
    ]]
    
    doctors = model_doc.predict(X_staff_input)[0]
    nurses = model_nurse.predict(X_staff_input)[0]
    surgeons = model_surg.predict(X_staff_input)[0]
    
    # =========================
    # STEP 6 — MODEL 1 (Surge)
    # =========================
    df['beds_occupied'] = predicted_beds
    df['utilization_rate'] = utilization
    
    X_clf_input = df[[
        'Total_Beds', 'Available_Beds',
        'beds_occupied',
        'ICU_Beds', 'Non_ICU_Beds',
        'Total_Doctors', 'Ventilators',
        'utilization_rate',
        'icu_ratio', 'nonicu_ratio',
        'doctor_bed_ratio', 'ventilator_ratio',
        'room_density', 'emergency_ratio',
        'City_enc', 'District_enc',
        'State_enc', 'City_Type_enc'
    ]]
    
    surge = clf.predict(X_clf_input)[0]
    
    # =========================
    # STEP 7 — MODEL 5 (Wait Time)
    # =========================
    
    # Estimate patients
    df['patients'] = (df['Total_Beds'] - df['Available_Beds']).clip(lower=1)
    
    df['patient_doctor_ratio'] = df['patients'] / df['Total_Doctors'].replace(0, 1)
    
    X_wait_input = df[[
        'Total_Beds',
        'Available_Beds',
        'Total_Doctors',
        'patients',
        'utilization',
        'patient_doctor_ratio',
        'icu_ratio',
        'room_density',
        'emergency_ratio',
        'City_enc',
        'District_enc',
        'State_enc',
        'City_Type_enc'
    ]]
    
    wait_time = wait_model.predict(X_wait_input)[0]
    wait_time = max(5, wait_time)
    
    # =========================
    # FINAL OUTPUT
    # =========================
    return {
        "Beds Occupied": round(predicted_beds),
        "Utilization (%)": round(utilization * 100, 2),
        "Doctors Needed": round(doctors),
        "Nurses Needed": round(nurses),
        "Surgeons Needed": round(surgeons),
        "Surge Status": "YES" if surge == 1 else "NO",
        "Wait Time (mins)": round(float(wait_time), 2)
    }

In [151]:
sample = {
    "Total_Beds": 150,
    "Available_Beds": 40,
    "ICU_Beds": 30,
    "Non_ICU_Beds": 120,
    "Total_Doctors": 3,
    "Ventilators": 10,
    "Total_Rooms": 80,
    "Emergency_Rooms": 10,
    "City": "Colaba",
    "District": "Mumbai City",
    "State": "Maharashtra",
    "City_Type": "Urban"
}

result = full_hospital_pipeline(sample)

for k, v in result.items():
    print(k, ":", v)

Beds Occupied : 81
Utilization (%) : 53.85
Doctors Needed : 7
Nurses Needed : 12
Surgeons Needed : 4
Surge Status : NO
Wait Time (mins) : 15.8
